# 05 — Sensibilidade a frequências: GmNet-S3 vs. StarNet-S4

Avalia os modelos **já treinados** (fine-tuning, 5 sementes cada; notebooks 02 e 03) nas
imagens de **teste filtradas** no domínio da frequência, para verificar se o GmNet depende
mais de informação de alta frequência (texturas, bordas) do que a StarNet. Não há treino.

Protocolo, fixado antes de ver resultados (`cerrado_flora.evaluation.frequency`,
`docs/decisoes.md`):
- pré-processamento de avaliação (lado menor 256 + recorte central 224); filtro no espaço de
  pixels, antes da normalização ImageNet; FFT 2D e máscara circular ideal de raio *r*;
- **passa-baixa** (só frequências ≤ *r*) e **passa-alta** (remove ≤ *r*, mantém a média de cada
  canal), *r* ∈ {4, 8, 16, 32, 64, 112}; 112 = Nyquist; mais a imagem original (13 condições);
- medida principal: **macro-F1 relativo por foto** = F1(filtrada) / F1(original), média ± desvio
  entre as 5 sementes; resumo de cada curva pela média entre os raios.

**Limitações**: imagens filtradas estão fora da distribuição de treino, então a comparação é
**entre modelos**, não em valores absolutos. GmNet-S3 e StarNet-S4 diferem em mais do que o
gating; uma diferença aqui não é atribuível só ao gating.

**Antes**: rode o `00_setup_colab.ipynb`. GPU recomendada (minutos); na CPU leva ~3 h.
Os resultados de cada modelo/semente vão para `<drive_root>/results/frequency/`; os já
calculados são reaproveitados.

In [ ]:
CONFIGS = {
    "GmNet-S3": "configs/experiments/gmnet_s3_ft.yaml",
    "StarNet-S4": "configs/experiments/starnet_s4_ft.yaml",
}
RUN_TESTS = True  # roda a suíte de testes (incluindo os de PyTorch) antes de avaliar
PER_SPECIES_CONDITION = "low_16"  # condição da análise por espécie (fixada antes dos resultados)

## 1. Ambiente

In [ ]:
import importlib
import sys
from pathlib import Path

import torch
import yaml

REPO_DIR = Path("/content/ProjetoFinalRNAP_UnB2026")
assert REPO_DIR.exists(), "Rode antes o 00_setup_colab.ipynb neste ambiente."

# O kernel em execução não lê o .pth da instalação editável (ver notebook 00).
SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
importlib.invalidate_caches()

from cerrado_flora.utils import run_metadata

paths = yaml.safe_load((REPO_DIR / "configs/paths.yaml").read_text())
cfgs = {name: yaml.safe_load((REPO_DIR / path).read_text()) for name, path in CONFIGS.items()}
# Regra 7: os dois modelos foram avaliados com o mesmo pré-processamento e os mesmos dados
data_cfgs = [{k: c["data"][k] for k in ("img_size", "resize")} for c in cfgs.values()]
assert all(d == data_cfgs[0] for d in data_cfgs)
data_cfg = data_cfgs[0]
datasets = {c["dataset"]["name"] for c in cfgs.values()}
assert len(datasets) == 1, "os dois modelos devem usar o mesmo dataset"

drive_root = Path(paths["drive_root"])
assert drive_root.exists(), "Google Drive não montado (seção 1 do notebook 00)."
images_root = Path(paths["local_cache"]) / datasets.pop()
assert (images_root / "images").exists(), "Dataset ausente: rode a seção 5 do notebook 00."
manifest = REPO_DIR / cfgs["GmNet-S3"]["dataset"]["manifest"]

meta = run_metadata.collect(REPO_DIR)
print("commit:", meta["git_commit"], "| alterações não commitadas:", meta["git_dirty"])
if meta["git_commit"] is None or meta["git_dirty"]:
    print("AVISO: sem commit limpo; estes resultados não devem entrar no artigo (modo git).")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp = device.type == "cuda"  # como na avaliação do treino
if device.type == "cpu":
    print("AVISO: sem GPU; a avaliação leva ~3 h na CPU do Colab (minutos numa T4).")

## 2. Testes

In [ ]:
import subprocess

if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    # -rfEs: resumo com falhas (f), erros (E) e testes pulados (s);
    # --tb=short: traceback curto de cada falha. Saída completa, sem cortes.
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-rfEs", "--tb=short", "tests"],
                       check=False, cwd=REPO_DIR, capture_output=True, text=True)
    print(r.stdout)
    if r.stderr:
        print(r.stderr)
    assert r.returncode == 0, "testes falharam (ver 'short test summary info' acima)"

## 3. Imagens de teste

Decodificadas uma vez (uint8, exato) e reaproveitadas por todos os modelos.

In [ ]:
import pandas as pd

from cerrado_flora.evaluation.frequency import conditions, load_images

test = load_images(manifest, images_root, data_cfg, split="test")
CONDS = conditions()
n_classes = int(pd.read_csv(manifest, usecols=["class_index"])["class_index"].max()) + 1
print(test["images"].shape, test["images"].dtype, "| classes:", n_classes, "| condições:", len(CONDS))

Exemplo do que o modelo vê em cada condição (para exibição, valores recortados em [0, 1]):

In [ ]:
import matplotlib.pyplot as plt

from cerrado_flora.evaluation.frequency import DEFAULT_RADII, apply_condition

x = test["images"][0:1].float() / 255.0
fig, axes = plt.subplots(2, len(DEFAULT_RADII) + 1, figsize=(2 * (len(DEFAULT_RADII) + 1), 4.3))
for row, kind in enumerate(("low", "high")):
    axes[row, 0].imshow(x[0].permute(1, 2, 0))
    axes[row, 0].set_title("original", fontsize=9)
    for j, r in enumerate(DEFAULT_RADII, start=1):
        y = apply_condition(x, kind, r)[0].clamp(0, 1)
        axes[row, j].imshow(y.permute(1, 2, 0))
        axes[row, j].set_title(f"{'passa-baixa' if kind == 'low' else 'passa-alta'} r={r}", fontsize=9)
for a in axes.flat:
    a.axis("off")
plt.tight_layout()
plt.show()

## 4. Avaliação nas condições filtradas

In [ ]:
import numpy as np

from cerrado_flora.evaluation.frequency import predict_conditions, summarize_conditions
from cerrado_flora.models.factory import build_model

freq_root = drive_root / paths["drive_subdirs"]["results"] / "frequency"
summaries, logits_all = [], {}
for model_name, cfg in cfgs.items():
    exp = cfg["name"]
    out_dir = freq_root / exp
    out_dir.mkdir(parents=True, exist_ok=True)
    for seed in cfg["seeds"]:
        csv_path, npz_path = out_dir / f"seed_{seed}.csv", out_dir / f"seed_{seed}.npz"
        if csv_path.exists() and npz_path.exists():
            print(f"{model_name} semente {seed}: já calculado")
        else:
            ckpt = drive_root / paths["drive_subdirs"]["checkpoints"] / exp / f"seed_{seed}" / "best.pt"
            model = build_model(cfg["model"], n_classes, None)  # pesos vêm do best.pt abaixo
            model.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=True))
            model.to(device)
            lg = predict_conditions(model, test["images"], CONDS, device, amp)
            np.savez_compressed(npz_path, **lg)
            summarize_conditions(lg, test["labels"], test["observation_ids"], n_classes) \
                .to_csv(csv_path, index=False)
            run_metadata.save(out_dir / f"seed_{seed}_metadata.json", extra={
                "experiment": exp, "seed": seed, "checkpoint": str(ckpt), "device": str(device),
                "amp": amp, "conditions": [list(c) for c in CONDS], "data": data_cfg})
            print(f"{model_name} semente {seed}: ok")
            del model
        df = pd.read_csv(csv_path)
        df.insert(0, "modelo", model_name)
        df.insert(1, "seed", seed)
        summaries.append(df)
        logits_all[(model_name, seed)] = dict(np.load(npz_path))
res = pd.concat(summaries, ignore_index=True)

### Conferência: sem filtro, os resultados devem coincidir com os do treino (`metrics.json`)

In [ ]:
import json

checks = []
for model_name, cfg in cfgs.items():
    for seed in cfg["seeds"]:
        m = json.loads((drive_root / paths["drive_subdirs"]["results"] / cfg["name"]
                        / f"seed_{seed}" / "metrics.json").read_text())
        for level in ("photo", "observation"):
            now = res.query("modelo == @model_name and seed == @seed and condicao == 'none' "
                            "and nivel == @level").iloc[0]
            checks.append({"modelo": model_name, "seed": seed, "nivel": level,
                           "macro_f1_treino": m["test"][level]["macro_f1"],
                           "macro_f1_agora": now["macro_f1"]})
checks = pd.DataFrame(checks)
checks["diferenca"] = (checks["macro_f1_agora"] - checks["macro_f1_treino"]).abs()
display(checks.round(4))
# Tolerância: a precisão mista pode trocar a predição de uma ou duas fotos
assert checks["diferenca"].max() < 0.005, "o carregamento não reproduz a avaliação do treino"

## 5. Curvas de sensibilidade

Macro-F1 por foto relativo à imagem original; faixa = ± 1 desvio padrão entre sementes.

In [ ]:
photo = res[res["nivel"] == "photo"]
base = photo[photo["tipo"] == "none"].set_index(["modelo", "seed"])["macro_f1"]
rel = photo[photo["tipo"] != "none"].copy()
rel["f1_relativo"] = rel["macro_f1"] / base.loc[list(zip(rel["modelo"], rel["seed"]))].to_numpy()

colors = {"GmNet-S3": "#2a78d6", "StarNet-S4": "#eb6834"}  # paleta categórica validada
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
titles = {"low": "Passa-baixa (só frequências ≤ r)", "high": "Passa-alta (remove frequências ≤ r)"}
for a, kind in zip(axes, ("low", "high")):
    for model_name in cfgs:
        g = rel[(rel["tipo"] == kind) & (rel["modelo"] == model_name)] \
            .groupby("raio")["f1_relativo"].agg(["mean", "std"])
        a.plot(g.index, g["mean"], color=colors[model_name], lw=2, marker="o", label=model_name)
        a.fill_between(g.index, g["mean"] - g["std"], g["mean"] + g["std"],
                       color=colors[model_name], alpha=0.2, lw=0)
    a.set_xscale("log", base=2)
    a.set_xticks(DEFAULT_RADII, [str(r) for r in DEFAULT_RADII])
    a.set_xlabel("raio r (índice de frequência; 112 = Nyquist)")
    a.set_title(titles[kind], loc="left")
    a.grid(color="#e5e4e0", lw=0.6)
    a.set_axisbelow(True)
    for side in ("top", "right"):
        a.spines[side].set_visible(False)
axes[0].set_ylabel("macro-F1 relativo (foto)")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()

Resumo de cada curva: média do F1 relativo entre os raios (área normalizada), por semente.

In [ ]:
auc = rel.groupby(["tipo", "modelo", "seed"])["f1_relativo"].mean().unstack("modelo")
display(auc.groupby("tipo").agg(["mean", "std"]).round(4))
diff = (auc["GmNet-S3"] - auc["StarNet-S4"]).rename("GmNet − StarNet").unstack("tipo")
print("Diferença por semente (pareamento só pelo número da semente; não são o mesmo modelo):")
display(diff.round(4))
display(diff.agg(["mean", "std"]).round(4))

Tabela completa (média ± desvio entre sementes), por foto e por observação:

In [ ]:
full = res.groupby(["nivel", "tipo", "raio", "modelo"], dropna=False)[["macro_f1", "top1"]] \
    .agg(["mean", "std"]).round(4)
full

## 6. Por espécie

Queda do F1 por classe (teste, por foto) na condição `PER_SPECIES_CONDITION`, média entre sementes. Quanto mais negativa, mais a espécie depende das frequências removidas.

In [ ]:
from cerrado_flora.evaluation.metrics import confusion_matrix, per_class

species = pd.read_csv(manifest.parent / "species.csv").dropna(subset=["class_index"])
names = species.set_index(species["class_index"].astype(int))["scientific_name"]

def class_f1(lg):
    return per_class(confusion_matrix(test["labels"], lg.argmax(axis=1), n_classes))["f1"].to_numpy()

drops = {}
for model_name in cfgs:
    seeds = [s for (m, s) in logits_all if m == model_name]
    drops[model_name] = np.mean([class_f1(logits_all[(model_name, s)][PER_SPECIES_CONDITION])
                                 - class_f1(logits_all[(model_name, s)]["none"]) for s in seeds], axis=0)
per_sp = pd.DataFrame(drops)
per_sp.insert(0, "especie", names.sort_index().to_numpy())
per_sp["suporte"] = np.bincount(test["labels"], minlength=n_classes)
per_sp.sort_values("GmNet-S3").round(3)